# Test 8: Qualitative Error Analysis of False Negatives

This notebook isolates the False Negatives (actual malware predicted as safe) from the GraphCodeBERT validation set, and prints the raw source code for the top 20 most confident mistakes.

**Goal**: Identify structural patterns in missed vulnerabilities (e.g., extremely short functions, specific CWEs, or SQL injections) to provide transparent qualitative analysis for the paper.

In [ ]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, SequentialSampler
from tqdm.auto import tqdm

In [ ]:
print("Loading cached dataset and raw texts...")
dataset = torch.load('../cached_dataset.pt')

with open('../final_dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Isolate Validation Split (same as Test 1 & 2)
val_dataset = torch.utils.data.Subset(dataset, range(143971, 143971+15996))
val_raw = raw_data[143971:143971+15996]
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
print("Loading GraphCodeBERT...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("microsoft/graphcodebert-base", num_labels=2)
model.load_state_dict(torch.load('../model.bin', map_location=device))
model.to(device)
model.eval()
print("Model loaded successfully!")

In [ ]:
eval_sampler = SequentialSampler(val_dataset)
eval_dataloader = DataLoader(val_dataset, sampler=eval_sampler, batch_size=32)

all_preds = []
all_labels = []
all_probs = []

print("Running inference on validation set...")
with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating"):
        inputs = {
            "input_ids": batch[0].to(device),
            "attention_mask": batch[1].to(device),
            "position_ids": batch[2].to(device),
            "labels": batch[3].to(device)
        }
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(inputs["labels"].cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

In [ ]:
false_negatives = []

for i, (pred, label, prob, raw) in enumerate(zip(all_preds, all_labels, all_probs, val_raw)):
    if label == 1 and pred == 0:
        false_negatives.append({
            'index': i,
            'confidence_safe': 1.0 - prob, # prob is the logit for malicious
            'code': raw['func'],
            'project': raw.get('project', 'Unknown'),
            'task_id': raw.get('task_id', 'Unknown')
        })

print(f"\nTotal False Negatives Found in Validation: {len(false_negatives)}")

# Sort by how confident the model was that it was SAFE (worst mistakes first)
false_negatives.sort(key=lambda x: x['confidence_safe'], reverse=True)

print("\n======================================================")
print("Analyzing the Top 10 Most Confident False Negatives:")
print("======================================================\n")

for i, fn in enumerate(false_negatives[:10]):
    print(f"[False Negative #{i+1}] - Originally from '{fn['project']}'")
    print(f"Model Confidence it was SAFE: {fn['confidence_safe'] * 100:.2f}%")
    print("-" * 40)
    lines = fn['code'].split('\n')
    
    # Print up to 25 lines of code to avoid massive spam
    max_lines = 25
    print('\n'.join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"... [Truncated {len(lines) - max_lines} more lines]")
    print("=" * 60 + "\n")